# 🚀 Modelo Baseline - LightGBM com Validação Cruzada 5-Fold
**Competição:** [Kaggle Playground Series Season 6 Episode 10](https://www.kaggle.com/competitions/playground-series-s6e10)  
**Métrica:** ROC AUC  

Este notebook treina um primeiro modelo baseline estruturado com **LightGBM**, utilizando validação cruzada estratificada (**5-Fold Stratified K-Fold**), calcula o score de validação local (OOF ROC AUC) e gera o arquivo final de submissão para o Kaggle.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from src.config import ID_COL, TARGET_COL, SUBMISSIONS_DIR, RANDOM_SEED
from src.data import load_train, load_test
from src.features import prepare_features
from src.models import get_lgbm_model
from src.evaluation import run_stratified_kfold_cv
from src.utils import seed_everything, save_submission

seed_everything(RANDOM_SEED)
print("Ambiente configurado com sucesso!")

## 1. Carregamento dos Dados

In [ ]:
train_df = load_train()
test_df = load_test()
test_ids = test_df[ID_COL].values

print(f"Treino: {train_df.shape}")
print(f"Teste:  {test_df.shape}")

## 2. Preparação de Atributos (Feature Preprocessing)
Codificamos variáveis categóricas usando codificação ordinal e separamos a variável dependente `satisfaction`.

In [ ]:
X_train, X_test, y_train, feature_names = prepare_features(
    train_df=train_df,
    test_df=test_df,
    create_domain_features=False,
    encode_categoricals=True,
)

print(f"Features preparadas ({len(feature_names)}): {feature_names}")
print(f"Distribuição do alvo em treino:\n{y_train.value_counts(normalize=True)}")

## 3. Treinamento e Validação Cruzada (5-Fold Stratified K-Fold)
Treinamos o LightGBM em cada fold, calculamos o score ROC AUC por fold e a média Out-Of-Fold (OOF).

In [ ]:
oof_preds, test_preds, metrics = run_stratified_kfold_cv(
    model_fn=get_lgbm_model,
    X=X_train,
    y=y_train,
    X_test=X_test,
    n_splits=5,
    seed=RANDOM_SEED,
)

print("\n--- RESULTADOS DA VALIDAÇÃO ---")
for fold, s in enumerate(metrics["fold_scores"]):
    print(f"Fold {fold + 1}: {s:.5f}")
print(f"Score OOF ROC AUC Geral: {metrics['overall_oof_auc']:.5f}")

## 4. Importância dos Atributos (Feature Importance)

In [ ]:
# Ajustar modelo completo para inspecionar importância
full_model = get_lgbm_model()
full_model.fit(X_train, y_train)

importances = pd.Series(full_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x=importances.values[:20], y=importances.index[:20], palette="mako")
plt.title("Top 20 Features Mais Importantes (LightGBM)")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

## 5. Geração do Arquivo de Submissão
Salvamos o arquivo `submission.csv` na pasta `submissions/` com as probabilidades preditas de satisfação.

In [ ]:
sub_path = save_submission(
    predictions=test_preds,
    test_ids=test_ids,
    filename=f"baseline_lgbm_auc_{metrics['overall_oof_auc']:.5f}.csv",
)
print(f"Arquivo pronto para envio: {sub_path}")